<a href="https://colab.research.google.com/github/Heptazero/ml-course-labs/blob/main/data-mining/elliptic-bitcoin-fraud-detection/03_graph_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric -q

from pathlib import Path

import torch
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from torch_geometric.datasets import EllipticBitcoinDataset
from sklearn.metrics import classification_report

DATA_ROOT = Path('/content/ml-course-labs-data/elliptic') if Path('/content').exists() else Path('data/elliptic')
dataset = EllipticBitcoinDataset(root=str(DATA_ROOT))
data = dataset[0]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
print("使用设备：", device)

第二格，定义模型结构，两层图 Transformer 叠起来，中间用 ReLU 激活：

In [ ]:
class GraphTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.2)
        self.conv2 = TransformerConv(hidden_channels * heads, out_channels, heads=1, dropout=0.2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GraphTransformer(in_channels=data.num_node_features, hidden_channels=64, out_channels=2).to(device)
print(model)

第三格，训练。跟之前决策树用 class_weight='balanced' 是同一个思路，这里手动算出两个类别的权重传给损失函数，防止模型偷懒只猜"正常"：

In [ ]:
import torch.optim as optim

train_labels = data.y[data.train_mask]
num_licit = (train_labels == 0).sum().item()
num_illicit = (train_labels == 1).sum().item()
class_weights = torch.tensor([1.0/num_licit, 1.0/num_illicit], dtype=torch.float)
class_weights = (class_weights / class_weights.sum() * 2).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(1, 101):
    loss = train()
    if epoch % 20 == 0:
        print(f"第{epoch}轮，loss={loss:.4f}")

评估，跟决策树/SVM 那格输出格式完全一样

In [ ]:
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

y_true = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask].cpu().numpy()

print(classification_report(y_true, y_pred, target_names=['正常(0)', '欺诈(1)']))